# LC2 — Crecimiento sano y diagnóstico de la variación
**Brightwell Partners · Online Retail II** · Tecsup 2026-I

**Declaración de uso de IA**: parte del código fue construida con asistencia de Kimi (Moonshot AI), revisada línea por línea por el equipo. Las decisiones metodológicas, la interpretación de resultados y la redacción del informe son responsabilidad del equipo.

| Integrante | Aporte principal |
|---|---|
| Anthony Henry Arias Zevallos | (Ejecución del Análisis de Cohortes) |
| Pedro Sebastian Alfieri Arteaga Guerra | (Auditoría y Preparación de Datos) |
| Piero Hideki Furushio Casanave | (Definición del Árbol de la Métrica North Star) |
| Renzo Sebastian Salazar Chavez | (Implementación de Métricas Guardrails) |
| Bryan Villazante Lopez | (Desarrollo del Puente de Variación (Variation Bridge)) |


## Bloque 0 — Entorno
**Decisión**: centralizar importaciones y constantes en una sola celda. Si algo falla aquí, el notebook no debe continuar: un resultado que depende de un entorno no reproducible no vale como evidencia ante el comité.

In [17]:
import hashlib
import requests
import polars as pl
import duckdb
import plotly.graph_objects as go

pl.Config.set_tbl_cols(25)   # sin esto Polars recorta columnas al imprimir y se pierden cifras de la auditoría

polars.config.Config

## Ejercicio 1 — Verificación de la fuente y periodo comparable
**Decisión**: comparar la huella SHA-256 *antes* de cargar el archivo. Si verificamos después de analizar, cualquier resultado queda contaminado por la duda sobre la fuente. El `assert` detiene la ejecución: preferimos un notebook que no corre a uno que analiza datos falsificados.

In [18]:
URL = "https://raw.githubusercontent.com/Rociosayan/analitica-empresarial-integrada/main/semana04/datos/online_retail_II.parquet"
HUELLA_PUBLICADA = "8f64c20d17d38ab02c0dfddda323574e0ee8c34e719df6ea574d16a6e3678e96"

# Descargamos como bytes (no como archivo) para hashear exactamente lo que viajó por la red
contenido = requests.get(URL, timeout=180).content
huella_observada = hashlib.sha256(contenido).hexdigest()

# Si la huella no coincide, nada de lo que sigue tiene valor probatorio: detenemos todo
assert huella_observada == HUELLA_PUBLICADA, f"INTEGRIDAD COMPROMETIDA: {huella_observada}"
print("Huella verificada:", huella_observada)
print("Registro íntegro: el archivo analizado es exactamente el publicado.")

# Guardamos una copia local fija: el resto del notebook lee siempre de aquí,
# así que un cambio futuro en la URL no altera los resultados de esta ejecución
with open("online_retail_II.parquet", "wb") as f:
    f.write(contenido)

Huella verificada: 8f64c20d17d38ab02c0dfddda323574e0ee8c34e719df6ea574d16a6e3678e96
Registro íntegro: el archivo analizado es exactamente el publicado.


**Decisión de nomenclatura**: renombramos solo `Customer ID` (por el espacio en el nombre, que complica toda sintaxis posterior) y creamos `Importe = Quantity × Price`. El enunciado advierte que el importe no viene dado: usar `Price` solo como proxy del valor de la línea subestimaría el negocio en exactamente la proporción de las cantidades.

In [19]:
df_crudo = pl.read_parquet("online_retail_II.parquet")

# Un solo renombre: "Customer ID" tiene un espacio que obligaría a escribir pl.col("Customer ID")
# en cada celda posterior; consolidar el nombre ahora evita errores silenciosos más adelante
df_crudo = df_crudo.rename({"Customer ID": "CustomerID"})

# El importe de línea NO existe en la fuente: se construye. Sin esta columna no se puede
# responder ninguna pregunta de dinero del laboratorio
df_crudo = df_crudo.with_columns(
    (pl.col("Quantity") * pl.col("Price")).alias("Importe")
)

print("Filas:", df_crudo.height, "| Columnas:", df_crudo.columns)
print("Rango de fechas:", df_crudo["InvoiceDate"].min(), "→", df_crudo["InvoiceDate"].max())

Filas: 1067371 | Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'Importe']
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


### Auditoría de calidad — medir ANTES de descartar
**Decisión metodológica central del ejercicio**: cada categoría problemática se cuenta en **filas y en importe**, sin eliminar nada todavía. Descartar primero y medir después es indistinguible de alterar el resultado (lo dice el propio enunciado). Las categorías se solapan entre sí (una cancelación puede tener cantidad negativa); por eso el descarte final se aplica como una regla única y secuencial, no como suma de categorías.

In [20]:
# Códigos que la documentación de la fuente y la exploración identifican como NO producto:
# portes (POST), envío manual (M), cargos bancarios, ajustes y donaciones. Tratarlos como
# ventas inflaría ingresos sin representar demanda real de catálogo
CODIGOS_NO_PRODUCTO = ["POST", "DOT", "M", "C2", "BANK CHARGES", "PADS", "CRUK"]

es_cancelacion   = pl.col("Invoice").str.starts_with("C")   # prefijo C = devolución/cancelación
es_no_producto   = pl.col("StockCode").is_in(CODIGOS_NO_PRODUCTO)

def filas_e_importe(df):
    # Devolvemos siempre la pareja (n_filas, importe): la consigna exige dinero, no solo recuento
    return df.height, round(df["Importe"].sum(), 2)

auditoria = {
    "filas_totales":              (df_crudo.height, round(df_crudo["Importe"].sum(), 2)),
    "filas_duplicadas_exactas":   filas_e_importe(df_crudo.filter(df_crudo.is_duplicated())),
    "devoluciones_cancelaciones": filas_e_importe(df_crudo.filter(es_cancelacion)),
    "cantidad_no_positiva":       filas_e_importe(df_crudo.filter(pl.col("Quantity") <= 0)),
    "precio_no_positivo":         filas_e_importe(df_crudo.filter(pl.col("Price") <= 0)),
    "sin_cliente_identificado":   filas_e_importe(df_crudo.filter(pl.col("CustomerID").is_null())),
    "codigo_no_producto":         filas_e_importe(df_crudo.filter(es_no_producto)),
}

for k, (n, imp) in auditoria.items():
    print(f"{k:30s} | filas: {n:>7,} | importe: {imp:>15,.2f} GBP")

filas_totales                  | filas: 1,067,371 | importe:   19,287,250.57 GBP
filas_duplicadas_exactas       | filas:  67,242 | importe:      857,785.64 GBP
devoluciones_cancelaciones     | filas:  19,494 | importe:   -1,526,667.86 GBP
cantidad_no_positiva           | filas:  22,950 | importe:   -1,527,041.43 GBP
precio_no_positivo             | filas:   6,207 | importe:     -158,676.14 GBP
sin_cliente_identificado       | filas: 243,007 | importe:    2,638,958.18 GBP
codigo_no_producto             | filas:   5,408 | importe:      322,045.51 GBP


### Regla de limpieza (una sola, secuencial)
Justificación de cada exclusión:
- **Cancelaciones (C)**: son devoluciones; su importe es negativo y compensa ventas del periodo, pero para medir *demanda* contamos ventas netas excluyéndolas y vigilándolas por separado (guardrail 1).
- **Cantidad o precio no positivos sin prefijo C**: registros anómalos que no representan venta.
- **Códigos no-producto**: portes y ajustes, no demanda de catálogo.
- **Sin CustomerID**: no se puede saber si el cliente vuelve; la North Star y las cohortes se caen sin identidad de cliente. Se descartan del análisis de clientes y se cuantifica su peso para declararlo como límite.
- **Duplicados exactos**: mismo registro repetido; conservar una sola copia.

In [21]:
filas_antes = df_crudo.height
importe_antes = df_crudo["Importe"].sum()

# La regla se aplica en un solo encadenamiento para que el orden de los filtros quede explícito
# y reproducible: cualquier fila que cumpla CUALQUIERA de las condiciones sale una sola vez
df_limpio = (
    df_crudo
    .filter(~es_cancelacion)                          # quitamos devoluciones: miden demanda neta, no bruta
    .filter(pl.col("Quantity") > 0)                   # cantidades nulas/negativas sin prefijo C: no son venta
    .filter(pl.col("Price") > 0)                      # precio cero/negativo: ajuste contable, no transacción
    .filter(~es_no_producto)                          # portes y cargos: no son demanda del catálogo
    .filter(pl.col("CustomerID").is_not_null())       # sin identidad de cliente no hay North Star ni cohortes
    .unique()                                         # duplicados exactos: una sola copia legítima
)

filas_despues = df_limpio.height
pct_descarte = (1 - filas_despues / filas_antes) * 100
# Importe "representado" por lo descartado = ventas brutas menos ventas limpias.
# Ojo: incluye importes negativos (cancelaciones), por eso se reporta con signo y se explica en 1.1
importe_descartado = importe_antes - df_limpio["Importe"].sum()

print(f"Descartado: {filas_antes - filas_despues:,} filas ({pct_descarte:.2f}% del registro)")
print(f"Diferencia de importe (bruto - limpio): {importe_descartado:,.2f} GBP")

Descartado: 290,742 filas (27.24% del registro)
Diferencia de importe (bruto - limpio): 2,213,774.39 GBP


### Ventanas comparables
**Decisión**: el registro cubre dic-2009 a dic-2011, pero arranca y cierra a mitad de mes. Comparar años calendario fabricaría variación inexistente. Tomamos dos ventanas de 12 meses completos:
- **P0**: dic-2009 → nov-2010
- **P1**: dic-2010 → nov-2011

Los registros de dic-2009 y dic-2011 quedan fuera **por diseño** y se cuentan con su razón. La verificación de completitud mensual dentro de cada ventana es obligatoria: un mes faltante interno invalidaría la comparación.

In [22]:
from datetime import date

INICIO_P0, FIN_P0 = date(2009, 12, 1), date(2010, 11, 30)
INICIO_P1, FIN_P1 = date(2010, 12, 1), date(2011, 11, 30)

# Truncamos la fecha a mes para tratar cada ventana como bloques cerrados de 12 meses
df_limpio = df_limpio.with_columns(pl.col("InvoiceDate").dt.truncate("1mo").alias("Mes"))

def asignar_periodo(mes):
    return (
        pl.when(mes.is_between(INICIO_P0, FIN_P0, closed="both")).then(pl.lit("P0"))
        .when(mes.is_between(INICIO_P1, FIN_P1, closed="both")).then(pl.lit("P1"))
        .otherwise(pl.lit("FUERA"))
        .alias("Periodo")
    )

df_limpio = df_limpio.with_columns(asignar_periodo(pl.col("Mes")))

# Fuera de ventana: se cuantifica y se justifica. Dic-2009 es arranque parcial del registro;
# dic-2011 es cierre parcial. NO se mezclan con la limpieza: son decisiones distintas
fuera = df_limpio.filter(pl.col("Periodo") == "FUERA")
print("Registros fuera de ambas ventanas:", fuera.height,
      "| importe:", round(fuera["Importe"].sum(), 2), "GBP")
print(fuera.group_by("Mes").agg([pl.len().alias("n"), pl.col("Importe").sum().alias("importe")]))

# Completitud: cada ventana debe tener exactamente 12 meses con registros
completitud = (
    df_limpio.filter(pl.col("Periodo") != "FUERA")
    .group_by(["Periodo", "Mes"]).agg(pl.len().alias("n"))
    .group_by("Periodo").agg(pl.len().alias("meses_con_datos"))
)
print(completitud)
assert completitud["meses_con_datos"].to_list() == [12, 12], "Ventana incompleta: revisar antes de continuar"

Registros fuera de ambas ventanas: 16964 | importe: 512228.08 GBP
shape: (1, 3)
┌─────────────────────┬───────┬───────────┐
│ Mes                 ┆ n     ┆ importe   │
│ ---                 ┆ ---   ┆ ---       │
│ datetime[ns]        ┆ u32   ┆ f64       │
╞═════════════════════╪═══════╪═══════════╡
│ 2011-12-01 00:00:00 ┆ 16964 ┆ 512228.08 │
└─────────────────────┴───────┴───────────┘
shape: (2, 2)
┌─────────┬─────────────────┐
│ Periodo ┆ meses_con_datos │
│ ---     ┆ ---             │
│ str     ┆ u32             │
╞═════════╪═════════════════╡
│ P0      ┆ 12              │
│ P1      ┆ 12              │
└─────────┴─────────────────┘


## Ejercicio 2 — North Star y árbol de métricas
**Decisión**: la North Star es el **ingreso mensual generado por clientes recurrentes** (clientes que compran en el mes y ya habían comprado antes).

¿Por qué no es de vanidad? Los ingresos totales suben aunque crezca solo el ruido (clientes de una sola compra, pedidos excepcionales). El ingreso recurrente solo sube si los clientes **vuelven**: es la definición misma de crecimiento sano para un negocio de repetición.

Árbol de drivers (producto exacto, sin aproximaciones):
- **Driver A** = número de clientes recurrentes del mes
- **Driver B** = ingreso medio por cliente recurrente del mes
- **North Star** = A × B

La reconstrucción se verifica con diferencia nula frente al ingreso real de recurrentes.

In [23]:
# Índice numérico de mes: permite restar meses (edad del cliente) sin trabajar con fechas
df_limpio = df_limpio.with_columns(
    (pl.col("Mes").dt.year() * 12 + pl.col("Mes").dt.month()).alias("MesIdx")
)

# Tabla cliente-mes con su gasto: la unidad mínima para hablar de recurrencia
gasto_cm = (
    df_limpio
    .group_by(["CustomerID", "Mes", "MesIdx"])
    .agg(pl.col("Importe").sum().alias("Gasto"))
)

# Primer mes de compra de cada cliente: define su cohorte y su condición de recurrente.
# Se calcula sobre TODO el limpio (no solo ventanas) para no retrasar artificialmente
# la primera compra de los clientes de P0
primera_compra = gasto_cm.group_by("CustomerID").agg(pl.col("MesIdx").min().alias("PrimeraIdx"))

recurrentes = (
    gasto_cm
    .join(primera_compra, on="CustomerID")
    # Edad 0 = mes de captación; exigir edad >= 1 excluye justo la primera compra:
    # un cliente que no ha vuelto aún no es recurrente, es sólo una esperanza
    .filter(pl.col("MesIdx") - pl.col("PrimeraIdx") >= 1)
)

# Drivers mensuales de la North Star
ns_mensual = (
    recurrentes
    .group_by("Mes")
    .agg([
        pl.len().alias("ClientesRecurrentes"),               # Driver A
        pl.col("Gasto").sum().alias("IngresoRecurrente"),    # North Star (valor objetivo)
    ])
    .with_columns(
        (pl.col("IngresoRecurrente") / pl.col("ClientesRecurrentes")).alias("IngresoMedioPorRecurrente")  # Driver B
    )
    .sort("Mes")
)

# Verificación: A × B debe reconstruir la North Star con error nulo (solo coma flotante)
ns_mensual = ns_mensual.with_columns(
    (pl.col("ClientesRecurrentes") * pl.col("IngresoMedioPorRecurrente")).alias("NS_reconstruida")
)
error = (ns_mensual["NS_reconstruida"] - ns_mensual["IngresoRecurrente"]).abs().max()
print(f"Error máximo de reconstrucción: {error:.10f} GBP  (debe ser ~0)")
assert error < 1e-6, "El árbol NO reconcilia: revisar definición de drivers"

# Serie del periodo más reciente (P1) con ambos drivers
print(ns_mensual.filter(pl.col("Mes") >= pl.lit(FIN_P0).dt.truncate("1mo")))

Error máximo de reconstrucción: 0.0000000001 GBP  (debe ser ~0)
shape: (14, 5)
┌──────────────┬──────────────────────┬───────────────────┬──────────────────────┬─────────────────┐
│ Mes          ┆ ClientesRecurrentes  ┆ IngresoRecurrente ┆ IngresoMedioPorRecur ┆ NS_reconstruida │
│ ---          ┆ ---                  ┆ ---               ┆ rente                ┆ ---             │
│ datetime[ns] ┆ u32                  ┆ f64               ┆ ---                  ┆ f64             │
│              ┆                      ┆                   ┆ f64                  ┆                 │
╞══════════════╪══════════════════════╪═══════════════════╪══════════════════════╪═════════════════╡
│ 2010-11-01   ┆ 1280                 ┆ 1.0045e6          ┆ 784.762648           ┆ 1.0045e6        │
│ 00:00:00     ┆                      ┆                   ┆                      ┆                 │
│ 2010-12-01   ┆ 808                  ┆ 538335.54         ┆ 666.256856           ┆ 538335.54       │
│ 00:00:00  

## Ejercicio 3 — Guardrails
**Principio**: un guardrail no se optimiza, se vigila. Cada uno lleva serie mensual, umbral declarado por el equipo (no una norma de la fuente) y la **decisión concreta** que se activa si se supera.

1. **Tasa de devolución mensual** = |importe de cancelaciones del mes| ÷ ventas brutas del mes. Calidad de la operación comercial: si crece, el crecimiento se está "devolviendo".
2. **Concentración top-10 clientes** = importe del mes de los 10 mayores clientes ÷ importe total del mes. Concentración de demanda: si crece, el negocio depende de un puñado de cuentas.

Ambos se calculan sobre la línea de tiempo completa (incluidos los meses parciales, porque el riesgo existe aunque el mes no entre a la comparación).

In [24]:
# GUARDRAIL 1 — devoluciones: se recalculan desde el CRUDO porque las eliminamos del análisis;
# un guardrail sobre datos limpios sería ciego por construcción
ventas_mes = (
    df_crudo.filter(~es_cancelacion).filter(pl.col("Quantity") > 0).filter(pl.col("Price") > 0)
    .with_columns(pl.col("InvoiceDate").dt.truncate("1mo").alias("Mes"))
    .group_by("Mes").agg(pl.col("Importe").sum().alias("VentasBrutas"))
)
devol_mes = (
    df_crudo.filter(es_cancelacion)
    .with_columns(pl.col("InvoiceDate").dt.truncate("1mo").alias("Mes"))
    .group_by("Mes").agg(pl.col("Importe").sum().abs().alias("Devoluciones"))
)

guardrail_devoluciones = (
    ventas_mes.join(devol_mes, on="Mes", how="left")
    .with_columns((pl.col("Devoluciones") / pl.col("VentasBrutas") * 100).alias("TasaDevolucionPct"))
    .sort("Mes")
)
print(guardrail_devoluciones.tail(12))

# GUARDRAIL 2 — concentración: sobre df_limpio (post-criterio de cliente identificado),
# porque la pregunta de riesgo es "cuánto dependemos de pocos clientes conocidos"
top10_mes = (
    df_limpio
    .group_by(["Mes", "CustomerID"]).agg(pl.col("Importe").sum().alias("Gasto"))
    .sort(["Mes", "Gasto"], descending=[False, True])
    .group_by("Mes").agg(pl.col("Gasto").head(10).sum().alias("Top10"))
)
total_mes = df_limpio.group_by("Mes").agg(pl.col("Importe").sum().alias("Total"))

guardrail_concentracion = (
    total_mes.join(top10_mes, on="Mes")
    .with_columns((pl.col("Top10") / pl.col("Total") * 100).alias("ConcTop10Pct"))
    .sort("Mes")
)
print(guardrail_concentracion.tail(12))

shape: (12, 4)
┌─────────────────────┬──────────────┬──────────────┬───────────────────┐
│ Mes                 ┆ VentasBrutas ┆ Devoluciones ┆ TasaDevolucionPct │
│ ---                 ┆ ---          ┆ ---          ┆ ---               │
│ datetime[ns]        ┆ f64          ┆ f64          ┆ f64               │
╞═════════════════════╪══════════════╪══════════════╪═══════════════════╡
│ 2011-01-01 00:00:00 ┆ 691364.56    ┆ 131364.3     ┆ 19.000728         │
│ 2011-02-01 00:00:00 ┆ 523631.89    ┆ 25569.24     ┆ 4.883056          │
│ 2011-03-01 00:00:00 ┆ 717639.36    ┆ 34372.28     ┆ 4.789631          │
│ 2011-04-01 00:00:00 ┆ 537808.621   ┆ 44601.5      ┆ 8.293192          │
│ 2011-05-01 00:00:00 ┆ 770536.02    ┆ 47202.51     ┆ 6.125932          │
│ …                   ┆ …            ┆ …            ┆ …                 │
│ 2011-08-01 00:00:00 ┆ 759138.38    ┆ 54333.75     ┆ 7.157292          │
│ 2011-09-01 00:00:00 ┆ 1.0586e6     ┆ 38902.55     ┆ 3.67494           │
│ 2011-10-01 00:00:00 ┆

**Umbrales y decisiones del equipo** (declarados como criterio propio):

| Guardrail | Umbral de alerta | Decisión si se supera |
|---|---|---|
| Tasa de devolución mensual | > 5% durante 2+ meses consecutivos | Pausar la valoración y exigir al vendedor auditoría de motivos de devolución antes de cerrar precio |
| Concentración top-10 clientes | > 40% del ingreso mensual | Condicionar la adquisición a un plan de diversificación de cartera o reducir el múltiplo ofrecido |

*El equipo los fija así porque un pico aislado es ruido, pero dos meses seguidos ya es patrón; y porque una cartera donde 10 cuentas mandan transfiere el riesgo de churn a la tesis de inversión.*

In [25]:
# Evaluación de umbrales: se cuenta cuántos meses los superan y se listan, porque
# "superar el umbral" sin contexto temporal no dispara ninguna decisión
UMBRA_DEVOL, UMBRA_CONC = 5.0, 40.0

alertas_devol = guardrail_devoluciones.filter(pl.col("TasaDevolucionPct") > UMBRA_DEVOL)
alertas_conc  = guardrail_concentracion.filter(pl.col("ConcTop10Pct") > UMBRA_CONC)

print("Meses con devolución > 5%:", alertas_devol.height, "→", alertas_devol["Mes"].to_list())
print("Meses con concentración > 40%:", alertas_conc.height, "→", alertas_conc["Mes"].to_list())

Meses con devolución > 5%: 16 → [datetime.datetime(2010, 3, 1, 0, 0), datetime.datetime(2010, 4, 1, 0, 0), datetime.datetime(2010, 5, 1, 0, 0), datetime.datetime(2010, 6, 1, 0, 0), datetime.datetime(2010, 8, 1, 0, 0), datetime.datetime(2010, 9, 1, 0, 0), datetime.datetime(2010, 10, 1, 0, 0), datetime.datetime(2010, 12, 1, 0, 0), datetime.datetime(2011, 1, 1, 0, 0), datetime.datetime(2011, 4, 1, 0, 0), datetime.datetime(2011, 5, 1, 0, 0), datetime.datetime(2011, 6, 1, 0, 0), datetime.datetime(2011, 7, 1, 0, 0), datetime.datetime(2011, 8, 1, 0, 0), datetime.datetime(2011, 10, 1, 0, 0), datetime.datetime(2011, 12, 1, 0, 0)]
Meses con concentración > 40%: 1 → [datetime.datetime(2011, 12, 1, 0, 0)]


## Ejercicio 4 — Descomposición de la variación
**Marco**: la variación de ingresos entre P0 y P1 se separa en:
- **Catálogo común** (productos vendidos en ambos periodos): se descompone en
  - *Efecto volumen* = (q₁−q₀)·p₀ → vendimos más/menos unidades al precio antiguo
  - *Efecto precio* = (p₁−p₀)·q₀ → cambió el precio sobre las unidades antiguas
  - *Efecto mezcla* = (q₁−q₀)·(p₁−p₀) → interacción; qué parte del cambio viene de la combinación volumen-precio
- **Productos nuevos** (solo P1): su ingreso suma.
- **Productos discontinuados** (solo P0): su ingreso resta.

**Por qué esta convención**: por producto, volumen + precio + mezcla = Δingreso exactamente (es una identidad algebraica), así que la reconciliación no depende de ninguna aproximación — se demuestra con un assert.

In [26]:
# Agregado por producto y periodo: el precio del periodo es el promedio ponderado por unidades
# (ingreso/cantidad), no el promedio simple de precios de línea, que se sesgaría con
# pedidos pequeños a precios atípicos
prod_periodo = (
    df_limpio.filter(pl.col("Periodo") != "FUERA")
    .group_by(["Periodo", "StockCode"])
    .agg([
        pl.col("Quantity").sum().alias("Cantidad"),
        pl.col("Importe").sum().alias("Ingreso"),
    ])
    .with_columns((pl.col("Ingreso") / pl.col("Cantidad")).alias("Precio"))
)

p0 = prod_periodo.filter(pl.col("Periodo") == "P0").drop("Periodo")
p1 = prod_periodo.filter(pl.col("Periodo") == "P1").drop("Periodo")

# Unión externa: un producto puede faltar en un periodo y eso ES información (nuevo/discontinuado)
catalogo = p0.join(p1, on="StockCode", how="full", suffix="_P1")

# Los tres efectos se calculan SOLO sobre el catálogo común (donde existen ambas caras)
comun = catalogo.filter(
    pl.col("Cantidad").is_not_null() & pl.col("Cantidad_P1").is_not_null()
).with_columns([
    ((pl.col("Cantidad_P1") - pl.col("Cantidad")) * pl.col("Precio")).alias("EfectoVolumen"),
    ((pl.col("Precio_P1") - pl.col("Precio")) * pl.col("Cantidad")).alias("EfectoPrecio"),
    ((pl.col("Cantidad_P1") - pl.col("Cantidad")) * (pl.col("Precio_P1") - pl.col("Precio"))).alias("EfectoMezcla"),
])

nuevos = catalogo.filter(pl.col("Cantidad").is_null())
discontinuados = catalogo.filter(pl.col("Cantidad_P1").is_null())

ing_p0 = p0["Ingreso"].sum()
ing_p1 = p1["Ingreso"].sum()
var_total = ing_p1 - ing_p0

ev, ep, em = comun["EfectoVolumen"].sum(), comun["EfectoPrecio"].sum(), comun["EfectoMezcla"].sum()
ing_nuevos = nuevos["Ingreso_P1"].sum()
ing_disc   = discontinuados["Ingreso"].sum()

reconstruida = ev + ep + em + ing_nuevos - ing_disc
print(f"Ingreso P0: {ing_p0:,.2f} | Ingreso P1: {ing_p1:,.2f} | Variación: {var_total:+,.2f}")
print(f"Volumen: {ev:+,.2f} | Precio: {ep:+,.2f} | Mezcla: {em:+,.2f}")
print(f"Nuevos: +{ing_nuevos:,.2f} | Discontinuados: -{ing_disc:,.2f}")
print(f"Suma de componentes: {reconstruida:+,.2f}  (debe igualar la variación observada)")

assert abs(reconstruida - var_total) < 0.01, "No reconcilia: el diagnóstico NO puede presentarse"
print("RECONCILIACIÓN EXACTA ✓")

Ingreso P0: 8,336,248.54 | Ingreso P1: 8,224,999.56 | Variación: -111,248.98
Volumen: -1,093,518.51 | Precio: +412,112.83 | Mezcla: -616,259.20
Nuevos: +1,844,055.86 | Discontinuados: -657,639.97
Suma de componentes: -111,248.98  (debe igualar la variación observada)
RECONCILIACIÓN EXACTA ✓


In [27]:
# Cascada: el paso visual del puente P0 → P1 que el comité leerá de un vistazo
fig = go.Figure(go.Waterfall(
    measure=["absolute", "relative", "relative", "relative", "relative", "relative", "total"],
    x=["Ingreso P0", "Volumen", "Mezcla", "Precio", "Productos nuevos", "Discontinuados", "Ingreso P1"],
    y=[ing_p0, ev, em, ep, ing_nuevos, -ing_disc, ing_p1],
    textposition="outside",
    connector={"line": {"color": "#9aa5b1"}},
    increasing={"marker": {"color": "#1B3A5C"}},
    decreasing={"marker": {"color": "#E74C3C"}},
    totals={"marker": {"color": "#7CB9E8"}},
))
fig.update_layout(title="Puente de variación de ingresos P0 → P1 (GBP)",
                  template="plotly_white", height=500)
fig.show()

## Ejercicio 5 — Cohortes y retención
**Reglas de construcción**:
- Cohorte = mes de **primera compra** del cliente (desde dic-2009, usando todo el limpio).
- Edad = meses transcurridos desde esa primera compra (antigüedad relativa, no calendario).
- Retención a edad k = % de la cohorte con compra en el mes cohorte+k, sobre el tamaño inicial de la cohorte.
- **Comparación honesta**: solo entran cohortes que ya alcanzaron la edad comparada (corte en edad 3 meses). Una celda vacía en una cohorte reciente es falta de información, no abandono — tratarla como cero invalida el ejercicio.
- Punto de corte del equipo: cohortes captadas en 2010 (antiguas) vs captadas en 2011 (recientes), comparadas a su mes 3.

In [28]:
ultimo_idx = df_limpio["MesIdx"].max()          # último mes con datos (dic-2011, parcial)
EDAD_CORTE = 3                                   # edad comparada declarada por el equipo

# Actividad cliente-mes enriquecida con su cohorte y edad relativa.
# gasto_cm ya existe (Ejercicio 2): se reutiliza para no recalcular la misma agregación
activos = gasto_cm.select(["CustomerID", "Mes", "MesIdx"]).join(primera_compra, on="CustomerID")
activos = activos.with_columns((pl.col("MesIdx") - pl.col("PrimeraIdx")).alias("Edad"))

# Tamaño de cada cohorte: cuántos clientes únicos fueron captados en ese primer mes
# (una primera compra por cliente, garantizado porque primera_compra ya agrupa por CustomerID)
tam_cohorte = (
    primera_compra
    .group_by("PrimeraIdx").agg(pl.len().alias("TamanoCohorte"))
)

# Retención por (cohorte, edad): clientes activos a esa edad / tamaño de la cohorte
retencion = (
    activos
    .group_by(["PrimeraIdx", "Edad"]).agg(pl.len().alias("Activos"))
    .join(tam_cohorte, on="PrimeraIdx")
    .with_columns((pl.col("Activos") / pl.col("TamanoCohorte") * 100).alias("RetencionPct"))
)

# Formato año-mes para leer la matriz
retencion = retencion.with_columns(
    (pl.lit("cohorte-") + (pl.col("PrimeraIdx") // 12).cast(pl.String) + "-"
     + (pl.col("PrimeraIdx") % 12).cast(pl.String).str.zfill(2)).alias("Cohorte")
)

matriz = retencion.pivot(values="RetencionPct", index="Cohorte", on="Edad").sort("Cohorte")
print(matriz)

shape: (25, 26)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬───┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Coh ┆ 17  ┆ 11  ┆ 13  ┆ 19  ┆ 12  ┆ 0   ┆ 9   ┆ 3   ┆ 14  ┆ 1   ┆ 8   ┆ 4   ┆ … ┆ 15  ┆ 5   ┆ 16  ┆ 20  ┆ 10  ┆ 6   ┆ 2   ┆ 18  ┆ 22  ┆ 21  ┆ 23  ┆ 24  │
│ ort ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆   ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- │
│ e   ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64 │
│ --- ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆   ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     │
│ str ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆   ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     │
╞═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════

In [29]:
# Comparación antiguas vs recientes a la edad de corte.
# Filtro de elegibilidad: la cohorte solo entra si su mes (corte) existe en los datos;
# esto excluye automáticamente las cohortes de 2011 que aún no cumplen 3 meses
elegibles = retencion.filter(
    (pl.col("PrimeraIdx") + EDAD_CORTE <= ultimo_idx) & (pl.col("Edad") == EDAD_CORTE)
).with_columns((pl.col("PrimeraIdx") // 12).alias("AnioCohorte"))

comparacion = (
    elegibles
    .group_by("AnioCohorte")
    .agg([
        pl.len().alias("N_Cohortes"),
        pl.col("RetencionPct").mean().alias("RetencionMediaMes3"),
    ])
    .sort("AnioCohorte")
)
print(f"Retención a los {EDAD_CORTE} meses, por año de captación:")
print(comparacion)

# Curvas de retención por año de cohorte (todas las edades observadas, solo cohortes elegibles)
curvas = (
    retencion
    .filter(pl.col("PrimeraIdx") + pl.col("Edad") <= ultimo_idx)
    .with_columns((pl.col("PrimeraIdx") // 12).alias("AnioCohorte"))
    .group_by(["AnioCohorte", "Edad"]).agg(pl.col("RetencionPct").mean().alias("RetencionMedia"))
)

fig2 = go.Figure()
for anio, grupo in curvas.group_by("AnioCohorte", maintain_order=True):
    fig2.add_trace(go.Scatter(x=grupo["Edad"], y=grupo["RetencionMedia"],
                              mode="lines+markers", name=f"Captados en {anio[0]}"))
fig2.update_layout(title="Curvas de retención por año de captación (cohortes elegibles)",
                   xaxis_title="Edad (meses desde primera compra)", yaxis_title="Retención (%)",
                   template="plotly_white")
fig2.show()

Retención a los 3 meses, por año de captación:
shape: (2, 3)
┌─────────────┬────────────┬────────────────────┐
│ AnioCohorte ┆ N_Cohortes ┆ RetencionMediaMes3 │
│ ---         ┆ ---        ┆ ---                │
│ i32         ┆ u32        ┆ f64                │
╞═════════════╪════════════╪════════════════════╡
│ 2010        ┆ 12         ┆ 23.172623          │
│ 2011        ┆ 10         ┆ 19.907758          │
└─────────────┴────────────┴────────────────────┘


## Ejercicio 6 — Segmentación explicativa y consulta reproducible
**Criterios elegidos** (distintos entre sí): **país de destino** (geografía) y **banda de gasto mensual del cliente** (tamaño del cliente: bajo/medio/alto por terciles en P0).

**Distinción obligatoria**: el segmento de mayor variación **en dinero** no es necesariamente el de mayor variación **porcentual** — una plaza pequeña puede caer 60% y no mover la aguja, mientras un mercado grande cae 8% y explica millones. Magnitud ≠ intensidad.

In [30]:
# --- Criterio 1: país ---
seg_pais = (
    df_limpio.filter(pl.col("Periodo") != "FUERA")
    .group_by(["Periodo", "Country"]).agg(pl.col("Importe").sum().alias("Ingreso"))
    .pivot(values="Ingreso", index="Country", on="Periodo")
    .fill_null(0)
    .with_columns([
        (pl.col("P1") - pl.col("P0")).alias("Variacion"),
        pl.when(pl.col("P0") > 0)
          .then((pl.col("P1") - pl.col("P0")) / pl.col("P0") * 100)
          .otherwise(None).alias("VariacionPct"),
    ])
)

# --- Criterio 2: banda de cliente por gasto mensual (terciles calculados en P0) ---
gasto_cliente_p0 = (
    df_limpio.filter(pl.col("Periodo") == "P0")
    .group_by("CustomerID").agg(pl.col("Importe").sum().alias("GastoP0"))
)
t1 = gasto_cliente_p0["GastoP0"].quantile(1/3)
t2 = gasto_cliente_p0["GastoP0"].quantile(2/3)

def banda(g):
    return (pl.when(g < t1).then(pl.lit("Bajo"))
             .when(g < t2).then(pl.lit("Medio"))
             .otherwise(pl.lit("Alto")).alias("BandaCliente"))

clientes_banda = gasto_cliente_p0.with_columns(banda(pl.col("GastoP0")))

seg_banda = (
    df_limpio.filter(pl.col("Periodo") != "FUERA")
    .join(clientes_banda.select(["CustomerID", "BandaCliente"]), on="CustomerID")
    .group_by(["Periodo", "BandaCliente"]).agg(pl.col("Importe").sum().alias("Ingreso"))
    .pivot(values="Ingreso", index="BandaCliente", on="Periodo")
    .fill_null(0)
    .with_columns([
        (pl.col("P1") - pl.col("P0")).alias("Variacion"),
        ((pl.col("P1") - pl.col("P0")) / pl.col("P0") * 100).alias("VariacionPct"),
    ])
)

print("=== Por país (top 10 por |variación|) ===")
print(seg_pais.with_columns(pl.col("Variacion").abs().alias("_a")).sort("_a", descending=True)
      .drop("_a").head(10))
print("\n=== Por banda de cliente ===")
print(seg_banda.sort("Variacion", descending=True))

=== Por país (top 10 por |variación|) ===
shape: (10, 5)
┌────────────────┬───────────┬───────────┬────────────┬──────────────┐
│ Country        ┆ P1        ┆ P0        ┆ Variacion  ┆ VariacionPct │
│ ---            ┆ ---       ┆ ---       ┆ ---        ┆ ---          │
│ str            ┆ f64       ┆ f64       ┆ f64        ┆ f64          │
╞════════════════╪═══════════╪═══════════╪════════════╪══════════════╡
│ United Kingdom ┆ 6.7735e6  ┆ 7.0488e6  ┆ -275226.26 ┆ -3.90461     │
│ Australia      ┆ 138103.81 ┆ 29696.2   ┆ 108407.61  ┆ 365.055495   │
│ EIRE           ┆ 250184.2  ┆ 331009.31 ┆ -80825.11  ┆ -24.417775   │
│ France         ┆ 177263.02 ┆ 125657.95 ┆ 51605.07   ┆ 41.067891    │
│ Japan          ┆ 37416.37  ┆ 5607.54   ┆ 31808.83   ┆ 567.251058   │
│ Denmark        ┆ 18060.44  ┆ 49211.35  ┆ -31150.91  ┆ -63.300255   │
│ Norway         ┆ 29708.94  ┆ 6120.72   ┆ 23588.22   ┆ 385.383092   │
│ Germany        ┆ 198356.98 ┆ 178038.09 ┆ 20318.89   ┆ 11.412665    │
│ Belgium        ┆ 3

### Consulta reproducible en DuckDB
**Por qué DuckDB sobre Polars ya calculado**: cualquier miembro del comité puede ejecutar el SQL sin abrir nuestro notebook, y contrastar ambos resultados demuestra que la consulta no es decorativa. Registramos el dataframe ya procesado y aplicamos los cinco requisitos: selección, WHERE con dos condiciones, CASE WHEN, GROUP BY y ORDER BY.

In [31]:
# DuckDB lee directamente dataframes de Polars presentes en memoria: registramos el de países
# con su variación ya calculada, para que el SQL opere sobre el MISMO número que Polars
duckdb.register("seg_pais", seg_pais.to_arrow())

consulta_comite = """
SELECT
    Country,
    ROUND(P0, 2) AS Ingreso_P0,
    ROUND(P1, 2) AS Ingreso_P1,
    ROUND(Variacion, 2) AS Variacion_GBP,
    CASE
        WHEN Variacion >= 0 THEN 'crece'
        ELSE 'cae'
    END AS Direccion
FROM seg_pais
WHERE P0 > 0
  AND ABS(Variacion) > 0
ORDER BY ABS(Variacion) DESC
LIMIT 10
"""

# La consulta corre contra la tabla registrada: si el comité replica esta sentencia
# en otro entorno con la misma tabla, obtiene exactamente este resultado
resultado_sql = duckdb.sql(consulta_comite).pl()

# Contraste con Polars: mismo filtro, mismo orden. Si difieren, la consulta no es reproducible
contraste_polars = (
    seg_pais
    .filter((pl.col("P0") > 0) & (pl.col("Variacion").abs() > 0))
    .with_columns(pl.when(pl.col("Variacion") >= 0).then(pl.lit("crece")).otherwise(pl.lit("cae")).alias("Direccion"))
    .sort(pl.col("Variacion").abs(), descending=True)
    .head(10)
    .select(["Country", "P0", "P1", "Variacion", "Direccion"])
)

# Verificación: igualdad de países, dirección y variación (tolerancia de redondeo del ROUND)
mismos_paises = resultado_sql["Country"].to_list() == contraste_polars["Country"].to_list()
diferencia = (resultado_sql["Variacion_GBP"].cast(pl.Float64)
              - contraste_polars["Variacion"].round(2)).abs().max()
print(resultado_sql)
print(f"\nContraste Polars vs SQL: mismos países ordenados = {mismos_paises}, "
      f"máx. diferencia por redondeo = {diferencia}")

shape: (10, 5)
┌────────────────┬────────────┬────────────┬───────────────┬───────────┐
│ Country        ┆ Ingreso_P0 ┆ Ingreso_P1 ┆ Variacion_GBP ┆ Direccion │
│ ---            ┆ ---        ┆ ---        ┆ ---           ┆ ---       │
│ str            ┆ f64        ┆ f64        ┆ f64           ┆ str       │
╞════════════════╪════════════╪════════════╪═══════════════╪═══════════╡
│ United Kingdom ┆ 7.0488e6   ┆ 6.7735e6   ┆ -275226.26    ┆ cae       │
│ Australia      ┆ 29696.2    ┆ 138103.81  ┆ 108407.61     ┆ crece     │
│ EIRE           ┆ 331009.31  ┆ 250184.2   ┆ -80825.11     ┆ cae       │
│ France         ┆ 125657.95  ┆ 177263.02  ┆ 51605.07      ┆ crece     │
│ Japan          ┆ 5607.54    ┆ 37416.37   ┆ 31808.83      ┆ crece     │
│ Denmark        ┆ 49211.35   ┆ 18060.44   ┆ -31150.91     ┆ cae       │
│ Norway         ┆ 6120.72    ┆ 29708.94   ┆ 23588.22      ┆ crece     │
│ Germany        ┆ 178038.09  ┆ 198356.98  ┆ 20318.89      ┆ crece     │
│ Belgium        ┆ 20065.83   ┆ 3563

## Cifras clave para las respuestas escritas
Ejecutar esta celda y **copiar los valores impresos** en las respuestas 1.1–6.1 y en el informe. Ninguna cifra se transcribe a mano de memoria: todo sale del cálculo.

In [32]:
# Centralizamos las cifras que las respuestas escritas deben citar, con etiquetas
# autoexplicativas para que cualquier integrante las ubique en el notebook
cifras = {
    "pct_registro_descartado": f"{pct_descarte:.2f}%",
    "importe_representado_descarte_GBP": f"{importe_descartado:,.2f}",
    "ventas_fuera_de_ventanas_filas": fuera.height,
    "variacion_total_GBP": f"{var_total:+,.2f}",
    "efecto_volumen_GBP": f"{ev:+,.2f}",
    "efecto_precio_GBP": f"{ep:+,.2f}",
    "efecto_mezcla_GBP": f"{em:+,.2f}",
    "ingreso_productos_nuevos_GBP": f"{ing_nuevos:,.2f}",
    "ingreso_discontinuados_GBP": f"{ing_disc:,.2f}",
    "meses_devolucion_sobre_umbral": alertas_devol.height,
    "meses_concentracion_sobre_umbral": alertas_conc.height,
    "retencion_media_antiguas_mes3": f"{comparacion.filter(pl.col('AnioCohorte') == 2010)['RetencionMediaMes3'][0]:.1f}%",
    "retencion_media_recientes_mes3": f"{comparacion.filter(pl.col('AnioCohorte') == 2011)['RetencionMediaMes3'][0]:.1f}%",
    "pais_mayor_variacion_abs": resultado_sql["Country"][0],
    "pais_mayor_variacion_abs_GBP": f"{resultado_sql['Variacion_GBP'][0]:+,.2f}",
}
for k, v in cifras.items():
    print(f"{k:40s} = {v}")

pct_registro_descartado                  = 27.24%
importe_representado_descarte_GBP        = 2,213,774.39
ventas_fuera_de_ventanas_filas           = 16964
variacion_total_GBP                      = -111,248.98
efecto_volumen_GBP                       = -1,093,518.51
efecto_precio_GBP                        = +412,112.83
efecto_mezcla_GBP                        = -616,259.20
ingreso_productos_nuevos_GBP             = 1,844,055.86
ingreso_discontinuados_GBP               = 657,639.97
meses_devolucion_sobre_umbral            = 16
meses_concentracion_sobre_umbral         = 1
retencion_media_antiguas_mes3            = 23.2%
retencion_media_recientes_mes3           = 19.9%
pais_mayor_variacion_abs                 = United Kingdom
pais_mayor_variacion_abs_GBP             = -275,226.26


## Respuestas escritas

**1.1** Se descartó 27.24% del registro (290,742 filas), que representaban 2,213,774.39 GBP de diferencia entre el importe bruto y el limpio. No altera la conclusión porque lo descartado se concentra en devoluciones, ajustes contables, duplicados y registros sin cliente identificado, es decir, en ruido operativo y no en demanda genuina: los ingresos limpios de ambas ventanas conservan la comparabilidad necesaria para medir la variación real del negocio.

**2.1** La North Star (ingreso mensual de clientes recurrentes) no es de vanidad porque solo sube si los clientes vuelven a comprar —su reconstrucción exacta como ClientesRecurrentes × IngresoMedioPorRecurrente cerró con un error de apenas 0.0000000001 GBP frente al ingreso real—, mientras que el ingreso total puede subir aunque nadie repita compra. Permite decidir si el crecimiento es sano antes de fijar el precio de adquisición.

**3.1** Sin guardrails, optimizar la North Star podría lograrse a costa de devoluciones crecientes (16 de los 24 meses observados ya superan el umbral de 5%) o de concentrar la demanda en pocos clientes (diciembre de 2011 llegó a 47.16% en el top 10, por encima del umbral de 40%): el negocio mostraría una métrica sana mientras se deteriora su calidad operativa y su base de clientes.

**4.1** El efecto volumen es el dominante (-1,093,518.51 GBP). Sin embargo, no explica por sí solo la variación neta (-111,248.98 GBP) porque los efectos se compensan entre sí: el efecto precio (+412,112.83 GBP) y, sobre todo, el ingreso de productos nuevos (+1,844,055.86 GBP) contrarrestan parcialmente la caída de volumen y la pérdida de productos discontinuados (-657,639.97 GBP), de modo que el resultado neto es mucho menor que cualquiera de sus componentes por separado.

**5.1** Las cohortes captadas en 2011 retienen peor a los 3 meses (19.9%) que las captadas en 2010 (23.2%). Si esa tendencia continúa, el fondo estaría pagando por una base de clientes que no vuelve a comprar: el precio de adquisición debería descender o condicionarse a un plan de retención verificable con metas concretas.

**6.1** Se prioriza Reino Unido por magnitud (-275,226.26 GBP, la mayor variación absoluta de todos los países), no por intensidad porcentual: su caída de -3.90% es modesta frente a países como Dinamarca (-63.30%) o EIRE (-24.42%), pero al representar la mayor parte del ingreso total, su deterioro pesa mucho más en dinero que caídas porcentuales grandes ocurridas en mercados pequeños.


## Informe de recomendación al comité de inversiones — Brightwell Partners

### Recomendación: **No adquirir** en los términos del argumento de venta actual

El argumento de venta sostiene que "los ingresos crecieron respecto del año anterior". Comparando dos ventanas de doce meses completos y comparables (P0: dic-2009–nov-2010 vs. P1: dic-2010–nov-2011), sobre datos verificados por huella SHA-256 y depurados de duplicados, devoluciones, ajustes y registros sin cliente, esa afirmación **no se sostiene**: el ingreso cayó de 8,336,248.54 GBP a 8,224,999.56 GBP, una variación de **-111,248.98 GBP (-1.33%)**.

### Cifras que sustentan la recomendación

1. **Variación total negativa**: -111,248.98 GBP entre P0 y P1, pese a la narrativa de crecimiento del vendedor.
2. **El deterioro es estructural, no superficial**: el efecto volumen del catálogo común —es decir, menos unidades vendidas de los productos que el negocio ya tenía— es de **-1,093,518.51 GBP**, el componente dominante de toda la descomposición. El ingreso total se sostiene casi exclusivamente por productos nuevos (**+1,844,055.86 GBP**), una fuente de ingreso sin historial de recurrencia comprobado.
3. **El golpe recae sobre los clientes más valiosos**: el segmento de clientes de mayor gasto ("Alto") perdió **-1,337,100 GBP aprox. (-19.09%)** entre P0 y P1, el mayor deterioro en dinero de los tres segmentos de gasto — no son los clientes marginales los que se están yendo.
4. **La retención empeora en las cohortes nuevas**: los clientes captados en 2011 retienen 19.9% a los 3 meses, frente a 23.2% de los captados en 2010 — el negocio recluta clientes que se quedan menos.
5. **Calidad operativa comprometida de forma crónica**: la tasa de devolución mensual supera el umbral de alerta (5%) en 16 de los 24 meses observados; no es un evento aislado, es un patrón sostenido.

### Principal riesgo

El "crecimiento" que se le vendió al fondo depende de introducir productos nuevos de forma continua para compensar una pérdida estructural de demanda en el catálogo y en la base de clientes existente. Si el ritmo de lanzamiento de productos nuevos se desacelera, o si esos productos no logran generar clientes recurrentes, el ingreso reportado no tiene sostén.

### Tres afirmaciones que la evidencia disponible NO permite sostener

- **No se puede afirmar que la empresa subió precios deliberadamente como estrategia**: el efecto precio (+412,112.83 GBP) mide el cambio agregado entre periodos, no la intención comercial detrás de él.
- **No se puede afirmar la rentabilidad ni proyectar utilidad futura del negocio**: el conjunto de datos solo registra transacciones de ingreso; no contiene costos, márgenes ni gastos operativos.
- **No se puede afirmar por qué las cohortes recientes retienen peor** (¿producto, servicio, competencia, entorno macroeconómico?): la matriz de cohortes describe el patrón observado, no explica su causa.

### Advertencia metodológica

Esta descomposición identifica **qué** componentes integran el cambio de ingresos (volumen, precio, mezcla, catálogo nuevo/discontinuado), pero no demuestra **por qué** ocurrieron. El informe distingue explícitamente entre evidencia observada (las cifras anteriores), inferencia razonable (la lectura de riesgo de sostenibilidad) y propuesta de gestión (las condiciones sugeridas a continuación).

### Si el fondo insiste en explorar la operación — condiciones mínimas

- Renegociar el precio de compra reflejando la caída real de -1.33%, no la narrativa de crecimiento del vendedor.
- Exigir un desglose de las causas de devolución y un plan verificable para bajar la tasa mensual por debajo de 5% de forma sostenida.
- Condicionar parte del pago (earn-out) a que las cohortes futuras alcancen una retención a 3 meses igual o superior al 23% observado en 2010.
- Exigir evidencia de que el ingreso de productos nuevos genera clientes recurrentes, y no solo compras únicas que infladan el ingreso de un periodo.
